# Этап 2.3. Подготовка данных

По методичке (п.4 и п.5):
1. предварительный анализ: объём, пропуски, смысл атрибутов, их значимость, примеры записей;
2. очистка;
3. аннотирование и векторизация текстов;
4. слияние: шкалы, роли, общие атрибуты;
5. выбор хранилища, скрипт слияния и обновления.

Весь код загрузки, очистки и слияния лежит в `etl.py`, ноутбук вызывает его функции и показывает результат. Период — 2019–2025 гг.

In [1]:
import sqlite3
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
import etl

START, END = '2019-01-01', '2025-12-31'

## 1. Загрузка

In [2]:
price = etl.load_price(START, END)          # JSON, Energy-Charts
power = etl.load_power(START, END)          # JSON, Energy-Charts
weather = etl.load_weather(START, END)      # JSON, Open-Meteo
holidays = etl.load_holidays(START, END)    # JSON, Nager.Date
eurostat = etl.load_eurostat(START, END)    # XML, Eurostat
opsd = etl.load_opsd()                      # SQLite, OPSD
news_raw = etl.load_news(START, END)        # текст, Google News

## 2. Предварительный анализ

### 2.1. Объём и пропуски

In [3]:
sources = {'цена': price.to_frame(), 'генерация': power, 'погода': weather, 'праздники': holidays,
           'Eurostat': eurostat, 'OPSD': opsd, 'новости': news_raw}
pd.DataFrame({
    'строк': {k: len(v) for k, v in sources.items()},
    'столбцов': {k: v.shape[1] for k, v in sources.items()},
    'пропусков, %': {k: round(100 * v.isna().mean().mean(), 2) for k, v in sources.items()},
})

,строк,столбцов,"пропусков, %"
цена,67995,1,0.00
генерация,245472,22,3.97
погода,306960,7,0.00
праздники,135,4,0.00
Eurostat,84,5,0.00
OPSD,17540,2,0.00
новости,1948,5,0.00


In [4]:
# где именно пропуски
power.isna().mean().mul(100).round(1)[lambda s: s > 0]

Nuclear                    38.8
Fossil coal-derived gas    48.6
dtype: float64

Пропуски есть только в двух рядах генерации:
- `Nuclear` (39%) — АЭС Германии закрыты 15.04.2023, после этой даты ряд не публикуется. Пропуск означает ноль.
- `Fossil coal-derived gas` (49%) — ряд публикуется не за весь период и для анализа не нужен.

### 2.2. Примеры записей и распределения

In [5]:
price.describe().round(1)

count    67995.0
mean        94.6
std         90.3
min       -500.0
25%         39.4
50%         78.2
75%        113.7
max        936.3
Name: price, dtype: float64

In [6]:
power[['Load', 'Residual load', 'Solar', 'Wind onshore', 'Fossil gas']].describe().round(0)

,Load,Residual load,Solar,Wind onshore,Fossil gas
count,245472.0,245472.0,245472.0,245472.0,245472.0
mean,54741.0,34097.0,6037.0,11749.0,5542.0
std,9691.0,13391.0,9376.0,9313.0,3060.0
min,27933.0,-6652.0,0.0,42.0,831.0
25%,46857.0,25150.0,0.0,4439.0,3145.0
50%,54609.0,34604.0,109.0,9046.0,4968.0
75%,62269.0,43347.0,9622.0,16897.0,7233.0
max,81614.0,75212.0,50385.0,47901.0,17719.0


In [7]:
weather.head(3)

,time,temperature_2m,wind_speed_100m,shortwave_radiation,cloud_cover,weather_code,city
0,2018-12-31 00:00:00+00:00,2.9,25.4,0.0,24,1,Berlin
1,2018-12-31 01:00:00+00:00,2.8,22.1,0.0,28,1,Berlin
2,2018-12-31 02:00:00+00:00,2.0,21.6,0.0,89,3,Berlin


In [8]:
news_raw.head(3)

,title,source,pubDate,link,query
0,Strompreise und EE-Anlagen in der Energiewirts...,PwC,"Mon, 28 Jan 2019 16:13:14 GMT",https://news.google.com/rss/articles/CBMigwFBV...,strompreis
1,Steigende Strompreise: So sparen Eigentümer - ...,Presseportal,"Thu, 31 Jan 2019 08:00:00 GMT",https://news.google.com/rss/articles/CBMiWEFVX...,strompreis
2,Smogalarm in Polen - Veraltete Kohleöfen qualm...,SRF,"Tue, 22 Jan 2019 08:00:00 GMT",https://news.google.com/rss/articles/CBMioAFBV...,strompreis


### 2.3. Атрибуты: смысл, шкала, роль, значимость

Гипотеза: температура, ветер, время суток и день недели позволяют предсказывать «дешёвые окна» точнее простых правил.

Шкалы — по типологии из 8 аналитических шкал:
- **описательная ветвь:** номинальная (ID), контентная;
- **смысловая ветвь:** категориальная, дихотомическая, порядковая;
- **измерительная ветвь:** интервальная, отношений, абсолютная.

| Атрибут | Смысл | Ед. | Шкала | Роль | Значимость для гипотезы |
|:--|:--|:--|:--|:--|:--|
| `link` | ссылка на новость | — | номинальная (ID) | ключ | — (ключ таблицы `news`) |
| `title` | заголовок новости | — | контентная | источник признаков | низкая — из него получаются тема и сигнал |
| `time` | час наблюдения (UTC) | дата-время | интервальная | ключ слияния | высокая — общий атрибут всех источников |
| `date`, `year` | дата и год по местному времени | — | интервальная | ключ слияния | средняя — связь с дневными и месячными данными |
| `price` | цена рынка на сутки вперёд | EUR/МВт·ч | отношений | целевая | высокая — из неё строится «дешёвое окно» |
| `is_cheap_window` | час из 25% самых дешёвых за сутки | да/нет | дихотомическая | целевая | высокая — то, что предсказываем |
| `load` | потребление | МВт | отношений | объект | высокая — спрос |
| `residual_load` | потребление минус ветер и солнце | МВт | отношений | объект | высокая — что покрывают газ и уголь |
| `res_share` | доля ВИЭ в покрытии потребления | % | отношений | фактор | высокая |
| `solar`, `wind_onshore`, `wind_offshore` | выработка солнечных и ветровых станций | МВт | отношений | фактор | высокая — дешёвая генерация |
| `gas`, `lignite`, `hard_coal` | выработка газовых и угольных станций | МВт | отношений | фактор | средняя — дорогая генерация задаёт цену |
| `nuclear`, `biomass`, `hydro` | прочая генерация | МВт | отношений | фактор | низкая — почти постоянна |
| `net_import` | сальдо перетоков с соседями (знак = направление) | МВт | отношений | фактор | средняя |
| `temperature` | температура воздуха | °C | интервальная | фактор | высокая (в гипотезе) — отопление |
| `wind_speed_100m` | ветер на высоте ветрогенератора | км/ч | отношений | фактор | высокая (в гипотезе) |
| `radiation` | солнечная радиация | Вт/м² | отношений | фактор | высокая — выработка СЭС |
| `cloud_cover` | облачность | % | отношений | фактор | низкая |
| `weather_code` | тип погоды по WMO | — | категориальная | фактор | низкая |
| `city` | город замера погоды | — | категориальная | разрез | — (до слияния) |
| `hour`, `dow`, `month` | час, день недели, месяц | — | порядковая | фактор | высокая (в гипотезе) |
| `day_type` | рабочий / выходной / праздник | — | категориальная | фактор | высокая — поведение потребителей |
| `is_holiday` | федеральный праздник | да/нет | дихотомическая | фактор | средняя — спрос как в выходной |
| `holiday_states` | число земель, где день выходной | шт. | абсолютная | фактор | средняя |
| `market_regime` | до кризиса / кризис 2021–22 / после | — | порядковая | фактор | средняя — сдвиг уровня цен |
| `eu_res_share`, `eu_fossil_share` | доли ВИЭ и ископаемого топлива за месяц | доля | отношений | фактор | низкая — сезонный фон |
| `topic` | тема новости | — | категориальная | фактор | низкая |
| `price_signal` | рост / нейтрально / снижение цен в заголовке | −1, 0, 1 | порядковая | фактор | низкая |
| `news_count` | число новостей за день | шт. | абсолютная | фактор | низкая — внешний фон |
| `news_signal`, `news_share_*` | средний сигнал и доли тем за день | доля | отношений | фактор | низкая |

## 3. Очистка

Что найдено при анализе и что с этим делаем:

| Проблема | Решение |
|:--|:--|
| с 01.10.2025 цена публикуется каждые 15 минут, генерация — всегда каждые 15 минут | приводим к часу средним значением |
| летнее и зимнее время: сутки из 23 и 25 часов | храним время в UTC, местное время — только для календарных признаков |
| пропуски `Nuclear` после закрытия АЭС | заполняем нулём |
| 22 ряда генерации, часть не нужна | оставляем 13 рядов, называем коротко |
| погода в 5 городах, а цена одна на всю зону | усредняем по городам; код погоды — максимальный (самое выраженное явление) |
| экстремальные цены −500 и 936 EUR/МВт·ч | оставляем: это реальные события в пределах биржевых лимитов (−500…4000) |

In [9]:
price_h = etl.clean_price(price)
power_h = etl.clean_power(power)
weather_h = etl.clean_weather(weather)

full_range = pd.date_range(price_h.index.min(), price_h.index.max(), freq='h')
print('часов в периоде:', len(full_range), '| пропущенных часов в цене:', len(full_range.difference(price_h.index)))
print('пропусков после очистки:', power_h.isna().sum().sum() + weather_h.isna().sum().sum())
power_h.head(3)

часов в периоде: 61368 | пропущенных часов в цене: 0
пропусков после очистки: 0


,load,residual_load,res_share,solar,wind_onshore,wind_offshore,gas,lignite,hard_coal,nuclear,biomass,hydro,net_import
2018-12-31 23:00:00+00:00,43273.500,19940.175,70.875,0.0,20199.95,3133.400,3372.825,6407.750,2920.425,8994.100,4787.900,1824.225,-9677.6
2019-01-01 00:00:00+00:00,41631.475,16592.100,77.750,0.0,22171.70,2867.675,3057.475,5441.075,2497.025,8528.675,4785.775,1834.300,-10022.0
2019-01-01 01:00:00+00:00,40084.725,14587.075,81.600,0.0,23037.70,2459.925,2803.100,5402.275,2314.450,7947.500,4742.325,1753.100,-11210.9


Проверка физических границ: значения вне допустимых диапазонов были бы ошибками.

In [10]:
pd.Series({
    'цена вне −500…4000': ((price_h < -500) | (price_h > 4000)).sum(),
    'потребление ≤ 0': (power_h['load'] <= 0).sum(),
    'температура вне −40…45 °C': ((weather_h['temperature'] < -40) | (weather_h['temperature'] > 45)).sum(),
    'отрицательный ветер или радиация': ((weather_h['wind_speed_100m'] < 0) | (weather_h['radiation'] < 0)).sum(),
})

цена вне −500…4000                  0
потребление ≤ 0                     0
температура вне −40…45 °C           0
отрицательный ветер или радиация    0
dtype: int64

Сверка источников: OPSD (SQLite) и Energy-Charts (JSON) пересекаются в 10.2018 – 09.2020.

In [11]:
check = opsd.join(price_h.rename('price_ec'), how='inner').join(power_h['load'].rename('load_ec'))
pd.DataFrame({
    'среднее расхождение': [(check.price - check.price_ec).abs().mean(), (check.load - check.load_ec).abs().mean()],
    'корреляция': [check.price.corr(check.price_ec), check.load.corr(check.load_ec)],
}, index=['цена, EUR/МВт·ч', 'потребление, МВт']).round(3)

,среднее расхождение,корреляция
"цена, EUR/МВт·ч",0.000,1.000
"потребление, МВт",1096.917,0.995


Цена совпадает полностью: оба источника берут её из ENTSO-E. Потребление отличается в среднем на ~1,1 ГВт (2%) при корреляции 0,995 — источники по-разному считают нагрузку, поэтому в работе используется один источник — Energy-Charts.

## 4. Текст: аннотирование и векторизация

**Очистка:** убираем название издания из заголовка, дубликаты и нерелевантные заголовки (без слов про электроэнергию, энергию, газ или цены).

**Аннотирование** — по словарям ключевых слов (`etl.TOPICS`, `etl.UP`, `etl.DOWN`):
- `topic` — тема: газ, ВИЭ и отрицательные цены, биржа, регулирование, тарифы для потребителей, прочее;
- `price_signal` — о чём заголовок: +1 рост цен, −1 снижение, 0 нейтрально.

In [12]:
news = etl.annotate_news(news_raw)
print(len(news_raw), '→', len(news), 'заголовков после очистки')
news[['title', 'topic', 'price_signal']].sample(8, random_state=2)

1948 → 1503 заголовков после очистки


,title,topic,price_signal
847,Strompreise steigen ab 2025 – Darauf müssen si...,тарифы для потребителей,1
1047,Agora Energiewende fordert Ablösung der einhei...,прочее,0
1492,Vonovia sorgt für solaren Mieterstrom in Berli...,ВИЭ и отрицательные цены,0
113,Die Stromkosten in Deutschland waren 2020 so h...,прочее,1
359,Strompreis-Explosion: Regierung beantragt eine...,регулирование,1
459,Tiwag entscheidet bis Ende März über neuen Str...,прочее,0
1898,Nebenkosten-Hammer: Mieter müssen 800 Euro für...,газ,0
911,"Henkel-Chef Knobel über Donald Trump, die Ampe...",прочее,0


In [13]:
pd.crosstab(news['topic'], news['price_signal'], margins=True)

price_signal,-1,0,1,All
topic,,,,
ВИЭ и отрицательные цены,65,98,26,189
биржа,3,22,5,30
газ,36,99,37,172
прочее,99,521,160,780
регулирование,18,56,31,105
тарифы для потребителей,56,129,42,227
All,277,925,301,1503


**Векторизация** — TF-IDF: каждый заголовок становится вектором весов слов. Вес высокий, если слово часто встречается в заголовке и редко в остальных.

In [14]:
stop_words = ['der', 'die', 'das', 'und', 'in', 'für', 'mit', 'auf', 'zu', 'von', 'im', 'den', 'des', 'ist',
              'nicht', 'ein', 'eine', 'sich', 'bei', 'an', 'aus', 'wie', 'was', 'so', 'es', 'um', 'am', 'dem', 'wird']
tfidf = TfidfVectorizer(stop_words=stop_words, min_df=5)
vectors = tfidf.fit_transform(news['title'])
print('матрица:', vectors.shape, '(заголовков × слов)')

# самые характерные слова каждой темы — средний вес TF-IDF
weights = pd.DataFrame(vectors.toarray(), columns=tfidf.get_feature_names_out(), index=news['topic'])
weights.groupby(level=0).mean().apply(lambda row: ', '.join(row.nlargest(6).index), axis=1)

матрица: (1503, 374) (заголовков × слов)


topic
ВИЭ и отрицательные цены    negative, strompreise, negativen, energien, wi...
биржа                       strombörse, strompreise, spotmarkt, cent, pro,...
газ                           gas, strom, gaspreise, gaspreis, preise, senken
прочее                      strompreis, strompreise, strom, deutschland, 2...
регулирование               habeck, strompreis, regierung, strompreise, ne...
тарифы для потребителей     strompreise, kunden, verbraucher, stromtarife,...
dtype: str

Для слияния с почасовыми данными тексты агрегируются по дням: число публикаций, средний ценовой сигнал и доли тем.

In [15]:
news_by_day = etl.news_daily(news)
news_by_day.head()

,news_count,news_signal,news_share_ВИЭ и отрицательные цены,news_share_биржа,news_share_газ,news_share_прочее,news_share_регулирование,news_share_тарифы для потребителей
date,,,,,,,,
2019-01-01,1,1.0,0.0,0.0,1.0,0.0,0.0,0.0
2019-01-28,1,0.0,0.0,0.0,0.0,1.0,0.0,0.0
2019-01-31,1,0.0,0.0,0.0,0.0,1.0,0.0,0.0
2019-02-25,1,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2019-03-04,1,0.0,0.0,0.0,0.0,0.0,1.0,0.0


## 5. Слияние

Источники имеют разную детализацию. Общий атрибут — время; для каждого источника определяем, как он переносится на часовую сетку:

| Источник | Формат | Детализация | Общий атрибут | Как переносится на час |
|:--|:--|:--|:--|:--|
| Energy-Charts: цена, генерация | JSON | 15 мин / час | время UTC | среднее за час |
| Open-Meteo | JSON | час × город | время UTC | среднее по городам |
| Nager.Date | JSON | день | местная дата | значение дня — на все его часы |
| Google News | текст (RSS/XML) | статья | местная дата | агрегат дня — на все его часы |
| Eurostat | XML | месяц | месяц | значение месяца — на все его часы |
| OPSD | SQLite | час | время UTC | только для сверки |

Дополнительно создаются календарные признаки, тип дня, режим рынка и целевой признак `is_cheap_window` — час из 25% самых дешёвых часов своих суток.

In [16]:
dataset = etl.build_dataset(price_h, power_h, weather_h, holidays, eurostat, news_by_day)
print(dataset.shape)
print('пропусков:', dataset.isna().sum().sum())
dataset.head(3)

(61368, 39)
пропусков: 0


,price,load,residual_load,res_share,solar,wind_onshore,wind_offshore,gas,lignite,hard_coal,...,news_count,news_signal,news_share_ВИЭ и отрицательные цены,news_share_биржа,news_share_газ,news_share_прочее,news_share_регулирование,news_share_тарифы для потребителей,is_cheap_window,market_regime
time,,,,,,,,,,,,,,,,,,,,,
2018-12-31 23:00:00+00:00,28.32,43273.500,19940.175,70.875,0.0,20199.95,3133.400,3372.825,6407.750,2920.425,...,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,False,до кризиса
2019-01-01 00:00:00+00:00,10.07,41631.475,16592.100,77.750,0.0,22171.70,2867.675,3057.475,5441.075,2497.025,...,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,False,до кризиса
2019-01-01 01:00:00+00:00,-4.08,40084.725,14587.075,81.600,0.0,23037.70,2459.925,2803.100,5402.275,2314.450,...,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,False,до кризиса


In [17]:
dataset.dtypes.value_counts()

float64     29
int32        4
bool         2
int64        1
object       1
str          1
category     1
Name: count, dtype: int64

## 6. Хранение

Объединённый набор — около 61 тыс. часов × 40 признаков (десятки МБ), плюс таблицы новостей, праздников и месячных данных.

| Вариант | Плюсы | Минусы |
|:--|:--|:--|
| CSV | просто | нет типов, индексов и обновления отдельных строк |
| Parquet | быстрое чтение столбцов | обновить часть строк нельзя — только перезаписать файл |
| PostgreSQL | промышленная СУБД, используется в EcoGrid | нужен отдельный сервер |
| **SQLite** | реляционная база в одном файле: SQL, индексы, транзакции, замена строк по ключу; встроена в Python | один пишущий процесс одновременно |

**Выбор — SQLite** (`data/energy.db`): объём небольшой, сервер не нужен, а обновление по ключу (час, ссылка на новость, дата, месяц) позволяет дописывать новые периоды и перезаписывать уточнённые. Таблицы при необходимости переносятся в PostgreSQL без изменения структуры.

| Таблица | Ключ | Содержимое |
|:--|:--|:--|
| `hourly` | `time` (час UTC) | объединённый набор |
| `news` | `link` | статьи с темой и ценовым сигналом |
| `holidays` | `date` | праздники |
| `eurostat_monthly` | `month` | производство по видам топлива |

In [18]:
etl.DB.unlink(missing_ok=True)   # строим базу заново
with sqlite3.connect(etl.DB) as con:
    etl.save(dataset, 'hourly', con, key='time')
    etl.save(news, 'news', con, key='link')
    etl.save(holidays, 'holidays', con, key='date')
    etl.save(eurostat, 'eurostat_monthly', con, key='month')
    print(pd.read_sql("SELECT name FROM sqlite_master WHERE type = 'table'", con))

               name
0            hourly
1              news
2          holidays
3  eurostat_monthly


In [19]:
# пример запроса к базе: средняя цена и доля часов с отрицательной ценой по типу дня
with sqlite3.connect(etl.DB) as con:
    display(pd.read_sql('''
        SELECT day_type, COUNT(*) AS hours, ROUND(AVG(price), 1) AS avg_price,
               ROUND(AVG(price < 0) * 100, 1) AS negative_price_pct
        FROM hourly GROUP BY day_type''', con))

,day_type,hours,avg_price,negative_price_pct
0,выходной,17280,74.2,7.1
1,праздник,1512,55.8,13.6
2,рабочий,42576,104.5,1.4


### Скрипт обновления

`etl.py` запускается из терминала с периодом. Он загружает все источники за этот период, чистит, объединяет и записывает в базу; строки с теми же ключами заменяются, поэтому повторный запуск не создаёт дублей:

```bash
python etl.py 2026-01-01 2026-01-31
```

Проверка — дописываем январь 2026 г. и запускаем ещё раз:

In [20]:
etl.update('2026-01-01', '2026-01-31')
etl.update('2026-01-01', '2026-01-31')   # повтор — дублей быть не должно

with sqlite3.connect(etl.DB) as con:
    display(pd.read_sql("SELECT year, COUNT(*) AS hours FROM hourly GROUP BY year", con))

,year,hours
0,2019,8760
1,2020,8784
2,2021,8760
3,2022,8760
4,2023,8760
5,2024,8784
6,2025,8760
7,2026,744


## Итог

1. **Загружено 7 источников в 4 форматах:** SQLite (OPSD), XML (Eurostat), JSON (Energy-Charts, Open-Meteo, Nager.Date) и текст (Google News).
2. **Предварительный анализ.** Пропуски есть только в двух рядах генерации, и оба объяснимы: АЭС закрыты, второй ряд неполный и не нужен. Значений вне физических границ нет. Цена в OPSD и Energy-Charts совпадает полностью, потребление расходится на 2%.
3. **Очистка.** Все ряды приведены к часу в UTC, пропуски устранены, выбраны 13 рядов генерации, погода сведена к одной зоне. После очистки пропусков нет.
4. **Текст.** Из 1948 заголовков после очистки осталось 1503. Каждый размечен темой (6 тем) и ценовым сигналом, векторизован TF-IDF (1503 × 374) и агрегирован по дням.
5. **Слияние.** Данные с детализацией 15 минут, час, день и месяц сведены в один почасовой набор: 61 368 часов × 39 признаков, без пропусков.
6. **Хранение.** SQLite `data/energy.db`, 4 таблицы с ключами. Скрипт `etl.py` дописывает и обновляет данные за любой период без дублей, это проверено на январе 2026 г.